### Perform Data wrangling of all the pilot data taken for LSSTWL-Y1

In [ ]:
from pathlib import Path
from collections import namedtuple

import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table, hstack, vstack,join
from astropy.coordinates import SkyCoord
import astropy.units as u
import pandas as pd
from scipy.integrate import trapezoid

import desispec.io
from desispec.spectra import stack

from prospect import utilities
import fitsio

In [ ]:
params = {
    "legend.fontsize": "x-large",
    "axes.labelsize": "x-large",
    "axes.titlesize": "x-large",
    "xtick.labelsize": "x-large",
    "ytick.labelsize": "x-large",
    "figure.facecolor": "w",
    "xtick.top": True,
    "ytick.right": True,
    "xtick.direction": "in",
    "ytick.direction": "in",
    "font.family": "serif",
    "mathtext.fontset": "dejavuserif"
}
plt.rcParams.update(params)

Observations were made in three fields, viz. COSMOS, XMM and HERCULES.

In [ ]:
#Storing all the file paths
fields = namedtuple("fields", ["name","spec_path", "fiber_assign_path","photometry_path","i_fibermag_lim","tert_masks"])

xmm = fields("XMMLSS","/global/cfs/cdirs/desi/users/raichoor/laelbg/daily/healpix/tertiary15-thru20221216/",
                  "/global/cfs/cdirs/desi/survey/fiberassign/special/tertiary/0015/tertiary-targets-0015-assign.fits",
                  "/global/cfs/cdirs/desi/users/bid13/DESI_II/target_data/HSC_XMM_I_mag_lim_24.8.fits",
             24.75, ["LSST_WLY1_HIP", "LSST_WLY1_LOP"])

cosmos = fields("COSMOS","/global/cfs/cdirs/desi/users/raichoor/laelbg/daily/healpix/tertiary23-thru20230326/",
                  "/global/cfs/cdirs/desi/survey/fiberassign/special/tertiary/0023/tertiary-targets-0023-assign.fits",
                  "/global/cfs/cdirs/desi/users/bid13/DESI_II/target_data/HSC_COSMOS_I_mag_lim_24.8.fits",
                25, ["LSST_WLY1_HIP", "LSST_WLY1_HIP2","LSST_WLY1_LOP"])

hercules = fields("HERCULES","/global/cfs/cdirs/desi/users/raichoor/laelbg/daily/healpix/tertiary27-thru20230428/",
                  "/global/cfs/cdirs/desi/survey/fiberassign/special/tertiary/0027/tertiary-targets-0027-assign.fits",
                  "/global/cfs/cdirs/desi/users/bid13/DESI_II/target_data/HSC_HERCULES_I_mag_lim_24.8.fits", 25, ["BRIGHT","FAINT"] )


out_path = Path("/global/cfs/cdirs/desi/users/bid13/DESI_II/pilot_obs/MERGED")

observations = [xmm,cosmos,hercules]

# observations = [xmm]

In [ ]:
conditions_xmm = Table.read(xmm.spec_path + "/exposures-daily-tertiary15-thru20221216.fits")
conditions_cosmos = Table.read(cosmos.spec_path + "/exposures-daily-tertiary23-thru20230326.fits")

In [ ]:
np.unique(conditions_xmm["NIGHT"])

In [ ]:
np.unique(conditions_cosmos["NIGHT"])

In [ ]:
print(f'XMM: Seeing (median +- 5th and 95th percentile): {np.median(conditions_xmm["SEEING_GFA"]):.2f}, {np.percentile(conditions_xmm["SEEING_GFA"],5):.2f}, {np.percentile(conditions_xmm["SEEING_GFA"],95):.2f}')
print(f'Cosmos: Seeing (median +- 5th and 95th percentile): {np.median(conditions_cosmos["SEEING_GFA"]):.2f}, {np.percentile(conditions_cosmos["SEEING_GFA"],5):.2f}, {np.percentile(conditions_cosmos["SEEING_GFA"],95):.2f}')

In [ ]:
print(f'XMM: airmass (median +- 5th and 95th percentile): {np.median(conditions_xmm["AIRMASS"]):.2f}, {np.percentile(conditions_xmm["AIRMASS"],5):.2f}, {np.percentile(conditions_xmm["AIRMASS"],95):.2f}')
print(f'Cosmos: airmass (median +- 5th and 95th percentile): {np.median(conditions_cosmos["AIRMASS"]):.2f}, {np.percentile(conditions_cosmos["AIRMASS"],5):.2f}, {np.percentile(conditions_cosmos["AIRMASS"],95):.2f}')

In [ ]:
print(f'XMM: airmass (median +- 5th and 95th percentile): {np.median(conditions_xmm["TRANSPARENCY_GFA"]):.2f}, {np.percentile(conditions_xmm["TRANSPARENCY_GFA"],5):.2f}, {np.percentile(conditions_xmm["TRANSPARENCY_GFA"],95):.2f}')
print(f'Cosmos: airmass (median +- 5th and 95th percentile): {np.median(conditions_cosmos["TRANSPARENCY_GFA"]):.2f}, {np.percentile(conditions_cosmos["TRANSPARENCY_GFA"],5):.2f}, {np.percentile(conditions_cosmos["TRANSPARENCY_GFA"],95):.2f}')

In [ ]:
plt.hist(conditions_xmm["SEEING_GFA"],histtype="step")
plt.hist(conditions_cosmos["SEEING_GFA"])
plt.xlabel("seeing")

In [ ]:
plt.hist(conditions_xmm["AIRMASS_GFA"],histtype="step")
plt.hist(conditions_cosmos["AIRMASS_GFA"])
plt.xlabel("airmass_GFA")

In [ ]:
plt.hist(conditions_xmm["AIRMASS"],histtype="step")
plt.hist(conditions_cosmos["AIRMASS"])
plt.xlabel("airmass")

In [ ]:
plt.hist(conditions_xmm["TRANSPARENCY_GFA"],histtype="step")
plt.hist(conditions_cosmos["TRANSPARENCY_GFA"])
plt.xlabel("airmass")

In [ ]:
conditions_xmm["TRANSPARENCY_GFA"].max()

In [ ]:
plt.hist(conditions_xmm["SEEING_GFA"],histtype="step")
plt.hist(conditions_cosmos["SEEING_GFA"])
plt.xlabel("seeing")

In [ ]:
def flux_to_mag(flux):
    return -2.5*np.log10(flux*1e-9) + 8.90

def prepare_photometry(file_path, i_fiber_mag_lim):
    hsc_cat = Table.read(file_path).to_pandas()
    # extinction corrected mags (extinction is negligible for XMM-LSS)
    hsc_cat["mag_i"] = flux_to_mag(hsc_cat["i_cmodel_flux"])-hsc_cat["a_i"]
    hsc_cat["mag_r"] = flux_to_mag(hsc_cat["r_cmodel_flux"])-hsc_cat["a_r"]
    hsc_cat["mag_z"] = flux_to_mag(hsc_cat["z_cmodel_flux"])-hsc_cat["a_z"]
    hsc_cat["mag_g"] = flux_to_mag(hsc_cat["g_cmodel_flux"])-hsc_cat["a_g"]
    hsc_cat["mag_y"] = flux_to_mag(hsc_cat["y_cmodel_flux"])-hsc_cat["a_y"]

    hsc_cat["mag_g_fiber"] = flux_to_mag(hsc_cat["g_fiber_flux"])-hsc_cat["a_g"]
    hsc_cat["mag_i_fiber"] = flux_to_mag(hsc_cat["i_fiber_flux"])-hsc_cat["a_i"]
    hsc_cat["mag_r_fiber"] = flux_to_mag(hsc_cat["r_fiber_flux"])-hsc_cat["a_r"]
    hsc_cat["mag_z_fiber"] = flux_to_mag(hsc_cat["z_fiber_flux"])-hsc_cat["a_z"]
    hsc_cat["mag_y_fiber"] = flux_to_mag(hsc_cat["y_fiber_flux"])-hsc_cat["a_y"]
    # hsc_cat["i_fiber_tot_mag"] = flux_to_mag(hsc_cat["i_fiber_tot_flux"])-hsc_cat["a_i"]

    ## Quality cuts
    # valid I-band flux
    qmask = np.isfinite(hsc_cat["i_cmodel_flux"]) & (hsc_cat["i_cmodel_flux"]>0)
    #cmodel fit not failed
    qmask &= (~hsc_cat["i_cmodel_flag"].values)
    #General Failure Flag
    qmask &= (~hsc_cat["i_sdsscentroid_flag"].values)

    # Possible cuts: Bright objects nearby, bad pixels

    #star-galaxy separation (is point source in I band)
    # mask &= (hsc_cat["i_extendedness_value"]>0)


    i_min = 22
    i_max = 24.5
    mask = (hsc_cat["mag_i"] <i_max) & (hsc_cat["mag_i"] >i_min)
    i_fiber_min = 22
    i_fiber_max = i_fiber_mag_lim #for cosmos and hercules the limit is 25 whereas for XMM it was set to 24.75
    mask &= (hsc_cat["mag_i_fiber"] <i_fiber_max) & (hsc_cat["mag_i_fiber"] >i_fiber_min)


    sels_cat = hsc_cat[qmask & mask]
    sels_cat = sels_cat.reset_index()
    
    return sels_cat

In [ ]:
spec_obj_list = []
rr_table_list = []
rr_details_list = []
tsnr_table_list = []
snr_table_list = []

fiber_assign_table_list = []
photometry_table_list =[]

for obs in observations:
    fiber_assign_cat = Table.read(obs.fiber_assign_path)
    mask = (np.isin(fiber_assign_cat["TERTIARY_TARGET"].astype(str), obs.tert_masks))
    mask &= (fiber_assign_cat['NASSIGN']>0)
    fiber_assign_cat = fiber_assign_cat[mask]
    fiber_assign_cat["FIELD_NAME"] = obs.name
    fiber_assign_table_list.append(fiber_assign_cat['TERTIARY_TARGET','TARGETID',"FIELD_NAME","NASSIGN"].to_pandas())
    photo_cat = prepare_photometry(obs.photometry_path, obs.i_fibermag_lim)
    photometry_table_list.append(photo_cat)
    

    for f in Path(obs.spec_path).glob("coadd*"):
        spectra = desispec.io.read_spectra(f)
        mask = spectra.fibermap["OBJTYPE"] == "TGT"
        mask &= spectra.fibermap["COADD_FIBERSTATUS"] ==0
        mask &= np.isin(spectra.fibermap["TARGETID"], fiber_assign_cat["TARGETID"])
        spectra = spectra[mask]
        if spectra.num_spectra()>0:
            snr_table = pd.DataFrame({"TARGETID":spectra.fibermap["TARGETID"]})
            for cam in "brz":
                snr_table[f"SNR_{cam.upper()}"] = np.median(spectra.flux[cam]*np.sqrt(spectra.ivar[cam]),axis=1)
            spec_obj_list.append(spectra)
            rr_table = Table.read(str(f).replace("coadd", "redrock"), hdu=1)
            tsnr_table = Table.read(str(f).replace("coadd", "redrock"), hdu=4)
            mask = np.isin(rr_table["TARGETID"], fiber_assign_cat["TARGETID"])
            rr_table_list.append(rr_table[mask])
            tsnr_table_list.append(tsnr_table[mask])

            redrock_det_file = str(f).replace("coadd", "rrdetails").replace("fits", "h5")
            rr_details_tab = utilities.match_rrdetails_to_spectra(redrock_det_file, spectra)
            rr_details_list.append(rr_details_tab)
            snr_table_list.append(snr_table)

In [ ]:
rr_table = vstack(rr_table_list)
rr_details_table = vstack(rr_details_list)
spectra = stack(spec_obj_list)
tsnr_table = vstack(tsnr_table_list)

fiber_assign_table = pd.concat(fiber_assign_table_list)
snr_table = pd.concat(snr_table_list)

In [ ]:
cat = join(spectra.fibermap, rr_table, keys="TARGETID", join_type="left")
cat = join(cat, tsnr_table, keys="TARGETID", join_type="left", uniq_col_name=None)
cat = join(cat, Table.from_pandas(fiber_assign_table), keys="TARGETID", join_type="left")
cat["EXPTIME"] = 12.15 * cat["TSNR2_LRG"]

In [ ]:
photometry_table = pd.concat(photometry_table_list)
spec_coord = SkyCoord(ra=cat["TARGET_RA"]*u.degree, dec = cat["TARGET_DEC"]*u.degree)

photo_coord = SkyCoord(ra=photometry_table["ra"].values*u.degree, dec=photometry_table["dec"].values*u.degree)

idx, d2d, d3d = spec_coord.match_to_catalog_sky(photo_coord)

phot_cat = photometry_table.iloc[idx]

cat = hstack([cat, Table.from_pandas(phot_cat)])

cat = cat[d2d.arcsec<0.000000001]

cat = join(cat, Table.from_pandas(snr_table), keys="TARGETID", join_type="left")
# cat["VI_quality"] = np.nan
# cat["best_z"] = np.nan

In [ ]:
print(len(cat))

In [ ]:
for col in cat.colnames:
    if cat[col].dtype == 'O':
        cat[col] = cat[col].value.astype(bool)
    if cat[col].dtype == '?':
        cat[col] = cat[col].value.astype(int)

In [ ]:
desispec.io.write_spectra(out_path / "spectra_LSST_WL_Y1.fits", spectra)
rr_table.write(out_path / "zbest_LSST_WL_Y1.fits",overwrite=True)
rr_details_table.write(out_path / "redrock_details_LSST_WL_Y1.fits",overwrite=True)
cat.write(out_path / "merged_cat_LSST_WL_Y1_no_VI.fits",overwrite=True)

# ADD VI in the merged catalog

In [ ]:
merged_cat = Table.read(out_path / "merged_cat_LSST_WL_Y1_no_VI.fits")

In [ ]:
vi_cats = []
vi_out_path = Path("/global/cfs/cdirs/desi/users/bid13/DESI_II/pilot_obs/DESI-II-VI-results")
for o in observations:
    vi_cats.append(pd.read_csv(vi_out_path / f"merged_vi_resolved_{o.name}.csv"))
vi_cat = pd.concat(vi_cats)

In [ ]:
# merged_cat.remove_columns(["VI_quality","best_z"])

In [ ]:
merged_cat = join(merged_cat,Table.from_pandas(vi_cat),keys="TARGETID",join_type="left")

In [ ]:
merged_cat.write(out_path / "merged_cat_LSST_WL_Y1.fits",overwrite=True)